# NANO-GPT 50M — максимум для T4
d_model 640, 10 слоёв, 10 голов, контекст 256, fp16 (AMP). Параметров ~49.6M.
Датасет тот же (131k пар). Батч 32, ~1.5-3 ч на T4.

**Перед запуском:** Среда выполнения → Сменить среду → GPU T4

In [ ]:
# Клон и проверка GPU
!git clone https://github.com/lucifermornngstar52-cell/nano-gpt.git
%cd nano-gpt
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader || echo 'GPU НЕ ВКЛЮЧЕНА!'

In [ ]:
# Обучение 50M (~1.5-3 ч на T4, чекпоинты каждые 500 шагов)
import os
os.environ.update(DM='640', NL='10', NH='10', CTX='256', BS='32', STEPS='6000')
# резюм при обрыве: os.environ['START'] = 'последний шаг из лога'
!python3 train_torch.py

In [ ]:
# Чат-тест
import torch, numpy as np
from train_torch import NanoT, vocab, stoi, dev, NL, T
model = NanoT().to(dev)
z = np.load('weights_big.npz')
with torch.no_grad():
    model.we.copy_(torch.tensor(z['we'])); model.wpe.copy_(torch.tensor(z['wpe']))
    for l in range(NL):
        b = model.blocks[l]
        for nm in ('q','k','v','o','fc','fc2'): getattr(b, nm).copy_(torch.tensor(z[f'{l}{nm}']))
        b.ln1.weight.copy_(torch.tensor(z[f'{l}ln1g'])); b.ln1.bias.copy_(torch.tensor(z[f'{l}ln1b']))
        b.ln2.weight.copy_(torch.tensor(z[f'{l}ln2g'])); b.ln2.bias.copy_(torch.tensor(z[f'{l}ln2b']))
    model.lnf.weight.copy_(torch.tensor(z['lnfg'])); model.lnf.bias.copy_(torch.tensor(z['lnfb']))
@torch.no_grad()
def reply(q, temp=0.8):
    idx = torch.tensor([[stoi.get(c,0) for c in ('В: '+q+'\nО:')[-T:]]], device=dev)
    out = ''
    for _ in range(120):
        lg = model(idx)[0,-1]/temp
        k = torch.topk(lg, 8); ni = k.indices[torch.multinomial(torch.softmax(k.values,-1),1)]
        ch = vocab[ni.item()]; out += ch
        if ch=='\n' and out.endswith('\n\n'): break
        idx = torch.cat([idx, ni.view(1,1)],1)
    return out.strip()
for q in ['как дела?','привет','расскажи о себе']:
    print('В:', q, '\nО:', reply(q), '\n')

In [ ]:
# Сохранить на Drive
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/nano-gpt && cp weights_big.npz meta.json /content/drive/MyDrive/nano-gpt/
print('в MyDrive/nano-gpt')